In [ ]:
"""
檔名：Q8dcard搜尋python廢文.py
功能：網頁資料擷取練習
學習重點：requests、BeautifulSoup、urllib3 網頁爬取與解析
"""
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

In [ ]:
options = Options()
options.binary_location = "/Applications/Brave Browser.app/Contents/MacOS/Brave Browser"

In [ ]:
driver = webdriver.Chrome(options=options)

In [ ]:
try:
    driver.get("https://www.dcard.tw/search/posts?query=Python")
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "article")))
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    articles = soup.find_all("article")

    print(f"共找到 {len(articles)} 篇文章，篩選廢文板：\n")

    count = 0
    for article in articles:
        try:
            lines = [
                l.strip()
                for l in article.get_text(separator="\n", strip=True).split("\n")
                if l.strip()
            ]

            # 找論壇名稱（通常是第一行含「板」的文字）
            forum = ""
            for line in lines[:5]:
                if "板" in line:
                    forum = line
                    break

            # 只保留廢文板
            if "廢文" not in forum:
                continue

            # 連結
            link_tag = article.find("a", href=lambda h: h and "/p/" in h)
            if not link_tag:
                continue
            href = link_tag.get("href", "")
            url = "https://www.dcard.tw" + href if not href.startswith("http") else href

            # 標題：最長的一行
            title = max(lines, key=len) if lines else "未知"

            # 按讚數：純數字中最大的
            nums = [int(l) for l in lines if l.isdigit()]
            likes = max(nums) if nums else 0

            count += 1
            print(f"標題：{title}")
            print(f"按讚：{likes}")
            print(f"連結：{url}")
            print()

        except:
            continue

    if count == 0:
        print("本頁搜尋結果無廢文板文章")
        print("\n所有文章的板名：")
        for article in articles:
            lines = [
                l.strip()
                for l in article.get_text(separator="\n", strip=True).split("\n")
                if l.strip()
            ]
            for line in lines[:5]:
                if "板" in line:
                    print(" ", line)
                    break

In [ ]:
finally:
    driver.quit()